# IIT-H Deep Learning Hackathon — Spring 2026
### CS5480 / AI5100 / AI2100 — Binary Image Classification
**No pretrained weights. Trained fully from scratch on Kaggle GPU.**

| Component | Details |
|---|---|
| Architecture | Custom SE-ResNet (~12.5M params) |
| Augmentation | Mixup, CutMix, ColorJitter, RandomErasing |
| Regularisation | Label Smoothing=0.05, Dropout, Weight Decay |
| LR Schedule | Linear Warmup + Cosine Annealing |
| Inference | EMA weights + 8x Test-Time Augmentation |
| Training | Progressive resize 64→96→128px, Mixed Precision |

## Cell 1: Verify Kaggle Dataset Paths

In [3]:
import os

# On Kaggle, dataset is auto-mounted here — no Drive needed!
DATA_DIR   = '/kaggle/input/competitions/iith-deep-learning-2026-hackathon'
OUTPUT_CSV = '/kaggle/working/predictions.csv'
CKPT_DIR   = '/kaggle/working'   # checkpoints saved here, persist during session

# Verify structure
train_path = os.path.join(DATA_DIR, 'train/train')
test_path  = os.path.join(DATA_DIR, 'test/test')

assert os.path.exists(train_path), f'ERROR: Missing {train_path}'
assert os.path.exists(test_path),  f'ERROR: Missing {test_path}'

classes = sorted(os.listdir(train_path))
print('Train path :', train_path)
print('Test path  :', test_path)
print('Classes    :', classes)
print('Test images:', len(os.listdir(test_path)))

for cls in classes:
    cls_path = os.path.join(train_path, cls)
    if os.path.isdir(cls_path):
        print(f'  Class {cls}: {len(os.listdir(cls_path))} images')

Train path : /kaggle/input/competitions/iith-deep-learning-2026-hackathon/train/train
Test path  : /kaggle/input/competitions/iith-deep-learning-2026-hackathon/test/test
Classes    : ['0', '1']
Test images: 5010
  Class 0: 9000 images
  Class 1: 9000 images


## Cell 2: Check GPU

In [2]:
import torch
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('VRAM           :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU! Go to Settings → Accelerator → GPU')

CUDA available : True
GPU            : Tesla T4
VRAM           : 15.6 GB


## Cell 3: Imports

In [4]:
import random, math, copy, csv, os
from collections import defaultdict

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

Device: cuda


## Cell 4: Hyperparameters & Seed

In [5]:
CFG = dict(
    img_size_stage1 = 64,
    img_size_stage2 = 96,
    img_size_final  = 128,
    batch_size      = 128,
    num_workers     = 4,
    epochs_stage1   = 40,
    epochs_stage2   = 30,
    epochs_stage3   = 40,
    lr              = 1e-3,
    weight_decay    = 1e-4,
    label_smoothing = 0.05,
    mixup_alpha     = 0.2,
    cutmix_alpha    = 0.5,
    ema_decay       = 0.999,
    tta_n           = 8,
    grad_clip       = 1.0,
    warmup_epochs   = 3,
    num_classes     = 2,
)

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark     = True

set_seed()
print('Config ready')

Config ready


## Cell 5: Model Architecture (SE-ResNet from scratch)

In [6]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, mid, bias=False),
            nn.SiLU(),
            nn.Linear(mid, channels, bias=False),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.se(x).view(x.size(0), x.size(1), 1, 1)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, dropout=0.0):
        super().__init__()
        self.bn1   = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.se    = SEBlock(out_ch)
        self.drop  = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()
        self.skip  = (nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False)
                      if (stride != 1 or in_ch != out_ch) else nn.Identity())

    def forward(self, x):
        out = self.conv1(F.silu(self.bn1(x)))
        out = self.drop(self.se(self.conv2(F.silu(self.bn2(out)))))
        return out + self.skip(x)


class CustomResNet(nn.Module):
    def __init__(self, num_classes=CFG['num_classes'], dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(16), nn.SiLU(),
            nn.Conv2d(16, 32, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.SiLU(),
        )
        self.maxpool = nn.MaxPool2d(3, stride=2, padding=1)
        self.layer1  = self._make(32,  64,  2, 1, 0.0)
        self.layer2  = self._make(64,  128, 2, 2, 0.1)
        self.layer3  = self._make(128, 256, 3, 2, 0.2)
        self.layer4  = self._make(256, 512, 2, 2, 0.2)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(512, 256), nn.SiLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(256, num_classes),
        )
        self._init_weights()

    @staticmethod
    def _make(ic, oc, n, s, d):
        return nn.Sequential(
            ResBlock(ic, oc, stride=s, dropout=d),
            *[ResBlock(oc, oc, dropout=d) for _ in range(n - 1)]
        )

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.maxpool(self.stem(x))
        return self.head(self.layer4(self.layer3(self.layer2(self.layer1(x)))))


# Sanity check
model = CustomResNet().to(DEVICE)
dummy = torch.randn(2, 3, 128, 128).to(DEVICE)
print('Output shape   :', model(dummy).shape)
print('Total params   :', f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Output shape   : torch.Size([2, 2])
Total params   : 12,560,722


## Cell 6: EMA, Dataset & Transforms

In [7]:
class ModelEMA:
    def __init__(self, model, decay=CFG['ema_decay']):
        self.ema   = copy.deepcopy(model).eval()
        self.decay = decay
        for p in self.ema.parameters(): p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for ep, mp in zip(self.ema.parameters(), model.parameters()):
            ep.copy_(ep * self.decay + mp.detach() * (1 - self.decay))
        for eb, mb in zip(self.ema.buffers(), model.buffers()):
            eb.copy_(mb)

    def __call__(self, x): return self.ema(x)


class HackDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, (-1 if label is None else label)


def load_train_samples(data_dir):
    samples = []
    train_dir = os.path.join(data_dir, 'train', 'train')
    for cls in sorted(os.listdir(train_dir)):
        cls_path = os.path.join(train_dir, cls)
        if not os.path.isdir(cls_path): continue
        for f in os.listdir(cls_path):
            if f.lower().endswith(('.jpg','.jpeg','.png','.bmp','.webp')):
                samples.append((os.path.join(cls_path, f), int(cls)))
    random.shuffle(samples)
    return samples


def load_test_samples(data_dir):
    test_dir = os.path.join(data_dir, 'test', 'test')
    exts = ('.jpg','.jpeg','.png','.bmp','.webp')
    files = sorted([f for f in os.listdir(test_dir) if f.lower().endswith(exts)])
    samples = [(os.path.join(test_dir, f), idx) for idx, f in enumerate(files)]
    print(f'Found {len(samples)} test images')
    return samples


def get_train_tfm(sz):
    return T.Compose([
        T.RandomResizedCrop(sz, scale=(0.7, 1.0)),
        T.RandomHorizontalFlip(),
        T.RandomRotation(10),
        T.ColorJitter(0.2, 0.2, 0.2, 0.05),
        T.RandomApply([T.GaussianBlur(3)], p=0.1),
        T.ToTensor(),
        T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        T.RandomErasing(p=0.1, scale=(0.02, 0.1)),
    ])

def get_val_tfm(sz):
    return T.Compose([
        T.Resize(int(sz * 1.1)),
        T.CenterCrop(sz),
        T.ToTensor(),
        T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])

def get_tta_tfms(sz):
    tfms = []
    for hf in [False, True]:
        for vf in [False, True]:
            ops = [T.Resize(int(sz*1.1)), T.CenterCrop(sz)]
            if hf: ops.append(T.RandomHorizontalFlip(p=1.0))
            if vf: ops.append(T.RandomVerticalFlip(p=1.0))
            ops += [T.ToTensor(),
                    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]
            tfms.append(T.Compose(ops))
    return tfms

print('Dataset utilities ready')

Dataset utilities ready


## Cell 7: Mixup / CutMix & Scheduler

In [9]:
def mixup(x, y, alpha=CFG['mixup_alpha']):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam*x + (1-lam)*x[idx], y, y[idx], lam

def cutmix(x, y, alpha=CFG['cutmix_alpha']):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    H, W = x.size(2), x.size(3)
    rh, rw = int(H*math.sqrt(1-lam)), int(W*math.sqrt(1-lam))
    cx, cy = random.randint(0,W), random.randint(0,H)
    x1,x2 = max(cx-rw//2,0), min(cx+rw//2,W)
    y1,y2 = max(cy-rh//2,0), min(cy+rh//2,H)
    xn = x.clone()
    xn[:,:,y1:y2,x1:x2] = x[idx,:,y1:y2,x1:x2]
    lam = 1 - (x2-x1)*(y2-y1)/(W*H)
    return xn, y, y[idx], lam

def mix_loss(crit, pred, ya, yb, lam):
    return lam*crit(pred,ya) + (1-lam)*crit(pred,yb)

def build_scheduler(opt, total_ep, warmup_ep, steps_per_ep):
    total  = total_ep  * steps_per_ep
    warmup = warmup_ep * steps_per_ep
    def fn(step):
        if step < warmup:
            return step / max(1, warmup)
        prog = (step - warmup) / max(1, total - warmup)
        return 0.5 * (1 + math.cos(math.pi * prog))
    return torch.optim.lr_scheduler.LambdaLR(opt, fn)

print('Augmentation & scheduler ready')

Augmentation & scheduler ready


## Cell 8: Training & Evaluation

In [10]:
def train_one_epoch(model, loader, opt, sched, crit, ema, scaler):
    model.train()
    tot_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        r = random.random()
        if r < 0.4:   imgs, ya, yb, lam = mixup(imgs, labels);   mixed = True
        elif r < 0.7: imgs, ya, yb, lam = cutmix(imgs, labels);  mixed = True
        else:         mixed = False

        opt.zero_grad()
        with torch.amp.autocast('cuda', enabled=DEVICE.type=='cuda'):
            logits = model(imgs)
            loss   = mix_loss(crit, logits, ya, yb, lam) if mixed else crit(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
        scaler.step(opt); scaler.update()
        sched.step(); ema.update(model)

        tot_loss += loss.item() * imgs.size(0)
        if not mixed:
            correct += (logits.argmax(1) == labels).sum().item()
            total   += imgs.size(0)

    return tot_loss / len(loader.dataset), (correct/total if total else float('nan'))


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        correct += (model(imgs).argmax(1) == labels).sum().item()
        total   += imgs.size(0)
    return correct / total


def train_stage(model, ema, train_s, val_s, img_size, n_ep, lr, ckpt_name):
    set_seed()

    # Save to /kaggle/working/ — persists during session
    ckpt_path = os.path.join(CKPT_DIR, ckpt_name)

    tr_ds  = HackDataset(train_s, get_train_tfm(img_size))
    val_ds = HackDataset(val_s,   get_val_tfm(img_size))

    tr_loader  = DataLoader(tr_ds,  CFG['batch_size'], shuffle=True,
                            num_workers=CFG['num_workers'], pin_memory=True,
                           persistent_workers=True)
    val_loader = DataLoader(val_ds, CFG['batch_size']*2, shuffle=False,
                            num_workers=CFG['num_workers'], pin_memory=True,
                           persistent_workers=True)

    opt    = optim.AdamW(model.parameters(), lr=lr, weight_decay=CFG['weight_decay'])
    sched  = build_scheduler(opt, n_ep, CFG['warmup_epochs'], len(tr_loader))
    crit   = nn.CrossEntropyLoss(label_smoothing=CFG['label_smoothing'])
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type=='cuda')

    best_acc, best_state = 0.0, None

    for ep in range(1, n_ep+1):
        tr_loss, tr_acc = train_one_epoch(model, tr_loader, opt, sched, crit, ema, scaler)
        val_acc_ema = evaluate(ema.ema, val_loader)
        val_acc_raw = evaluate(model, val_loader)
        val_acc = max(val_acc_ema, val_acc_raw)

        if val_acc > best_acc:
            best_acc   = val_acc
            best_state = copy.deepcopy(ema.ema.state_dict())
            torch.save(best_state, ckpt_path)   # save immediately on every improvement

        if ep % 5 == 0 or ep == n_ep:
            print(f'  Epoch {ep:3d}/{n_ep} | loss {tr_loss:.4f} '
                  f'| tr {tr_acc:.4f} '
                  f'| val_ema {val_acc_ema:.4f} | val_raw {val_acc_raw:.4f} '
                  f'| best {best_acc:.4f} '
                  f'| lr {opt.param_groups[0]["lr"]:.2e}')

    ema.ema.load_state_dict(best_state)
    print(f'  => Best val acc @ {img_size}px: {best_acc:.4f}')
    print(f'  => Checkpoint saved: {ckpt_path}')
    return best_acc

print('Training functions ready')

Training functions ready


## Cell 9: TTA Inference

In [11]:
@torch.no_grad()
def predict_tta(model, test_samples, img_size, tta_n=CFG['tta_n']):
    model.eval()
    all_probs = []
    for i, tfm in enumerate(get_tta_tfms(img_size)[:tta_n]):
        ds = HackDataset([(p, None) for p,_ in test_samples], tfm)
        loader = DataLoader(ds, CFG['batch_size']*2, shuffle=False,
                            num_workers=CFG['num_workers'], pin_memory=True,
                           persistent_workers=True)
        probs = []
        for imgs, _ in loader:
            imgs = imgs.to(DEVICE)
            probs.append(F.softmax(model(imgs), dim=1).cpu().numpy())
        all_probs.append(np.concatenate(probs))
        print(f'  TTA pass {i+1}/{tta_n} done')

    avg   = np.mean(all_probs, axis=0)
    preds = avg.argmax(axis=1)
    ids   = [s[1] for s in test_samples]
    return ids, preds

print('TTA inference ready')

TTA inference ready


## Cell 10: generate_predictions (competition required format)

In [13]:
# data_dir
# - train
#   -- 0
#   -- 1
# - test

def generate_predictions(data_dir, output_csv=OUTPUT_CSV):
    set_seed()
    print(f'Device : {DEVICE}')

    # Load data
    all_train = load_train_samples(data_dir)
    test_samp = load_test_samples(data_dir)

    # 90/10 stratified split
    by_class = defaultdict(list)
    for s in all_train: by_class[s[1]].append(s)
    train_s, val_s = [], []
    for cls, samps in by_class.items():
        random.shuffle(samps)
        cut = max(1, int(0.9 * len(samps)))
        train_s.extend(samps[:cut])
        val_s.extend(samps[cut:])

    print(f'Train: {len(train_s)} | Val: {len(val_s)} | Test: {len(test_samp)}')
    print(f'Class dist: { {c: sum(1 for s in train_s if s[1]==c) for c in by_class} }')

    # Build model + EMA
    model = CustomResNet(num_classes=CFG['num_classes']).to(DEVICE)
    ema   = ModelEMA(model)
    print(f'Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

    # Progressive training stages
    stages = [
        (CFG['img_size_stage1'], CFG['epochs_stage1'], CFG['lr'],       'ckpt_s1.pt'),
        (CFG['img_size_stage2'], CFG['epochs_stage2'], CFG['lr']*0.3,   'ckpt_s2.pt'),
        (CFG['img_size_final'],  CFG['epochs_stage3'], CFG['lr']*0.1,   'ckpt_s3.pt'),
    ]

    for sz, n_ep, lr, ckpt in stages:
        print(f'\n=== Stage {sz}x{sz} | {n_ep} epochs | lr={lr:.2e} ===')
        train_stage(model, ema, train_s, val_s, sz, n_ep, lr, ckpt)

    # TTA predictions
    print('\n=== Generating TTA Predictions ===')
    ids, preds = predict_tta(ema.ema, test_samp, CFG['img_size_final'])

    # Write CSV
    with open(output_csv, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['ID', 'TARGET'])
        for img_id, pred in zip(ids, preds):
            w.writerow([img_id, int(pred)])

    print(f'\nDone! Predictions: {output_csv}  ({len(ids)} rows)')

    # Show distribution
    unique, counts = np.unique(preds, return_counts=True)
    for u, c in zip(unique, counts):
        print(f'  Class {u}: {c} predictions ({100*c/len(preds):.1f}%)')

    return output_csv

print('generate_predictions() defined')

generate_predictions() defined


## Cell 11: RUN TRAINING

In [19]:
generate_predictions(DATA_DIR, OUTPUT_CSV)

Device : cuda
Found 5010 test images
Train: 16200 | Val: 1800 | Test: 5010
Class dist: {0: 8100, 1: 8100}
Parameters: 12,560,722

=== Stage 64x64 | 40 epochs | lr=1.00e-03 ===
  Epoch   5/40 | loss 0.3603 | tr 0.9604 | val_ema 0.5000 | val_raw 0.9889 | best 0.9889 | lr 9.93e-04
  Epoch  10/40 | loss 0.3357 | tr 0.9797 | val_ema 0.5756 | val_raw 0.9900 | best 0.9961 | lr 9.14e-04
  Epoch  15/40 | loss 0.2797 | tr 0.9898 | val_ema 0.8433 | val_raw 0.9978 | best 0.9989 | lr 7.62e-04
  Epoch  20/40 | loss 0.2962 | tr 0.9884 | val_ema 0.9706 | val_raw 0.9989 | best 0.9994 | lr 5.64e-04
  Epoch  25/40 | loss 0.3018 | tr 0.9922 | val_ema 0.9900 | val_raw 0.9994 | best 0.9994 | lr 3.54e-04
  Epoch  30/40 | loss 0.2689 | tr 0.9951 | val_ema 0.9989 | val_raw 0.9994 | best 0.9994 | lr 1.70e-04
  Epoch  35/40 | loss 0.2749 | tr 0.9936 | val_ema 0.9989 | val_raw 0.9989 | best 0.9994 | lr 4.44e-05
  Epoch  40/40 | loss 0.2438 | tr 0.9965 | val_ema 0.9994 | val_raw 0.9989 | best 0.9994 | lr 0.00e+00


'/kaggle/working/predictions.csv'

**Rebuild The Submission File**

In [18]:
# Rebuild model and load saved checkpoint
import zipfile

zip_path = '/kaggle/input/datasets/khwajaabdulsamad/checkpoint-1'  # your zip
extract_path = '/kaggle/working/ckpt' 

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
model = CustomResNet(num_classes=CFG['num_classes']).to(DEVICE)
ema = ModelEMA(model)

# Load the best checkpoint from stage 3
ema.ema.load_state_dict(torch.load('/kaggle/input/datasets/khwajaabdulsamad/checkpoint-1', map_location=DEVICE))
ema.ema.eval()
print('Checkpoint loaded successfully!')

# Now load fixed test samples
test_samp_fixed = load_test_samples_fixed(DATA_DIR)

# TTA with loaded model
ids, preds = predict_tta(ema.ema, test_samp_fixed, CFG['img_size_final'])

# Overwrite CSV
import csv
with open(OUTPUT_CSV, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['ID', 'TARGET'])
    for img_id, pred in zip(ids, preds):
        w.writerow([img_id, int(pred)])

print(f'\nDone! Fixed predictions saved to {OUTPUT_CSV}')
for i in range(5):
    print(f'  ID={ids[i]}, TARGET={preds[i]}')

IsADirectoryError: [Errno 21] Is a directory: '/kaggle/input/datasets/khwajaabdulsamad/checkpoint-1'

In [43]:
# Regenerate CSV with correct format
test_dir = os.path.join(DATA_DIR, 'test', 'test')
exts = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
files = sorted([f for f in os.listdir(test_dir) if f.lower().endswith(exts)])

test_samp_fixed = [(os.path.join(test_dir, f), f) for f in files]  # full filename as ID
# print(f'Found {len(test_samp_fixed)} test images')
# print('Sample:', test_samp_fixed[:3])

# TTA predictions
ids, preds = predict_tta(ema.ema, test_samp_fixed, CFG['img_size_final'])

# Write CSV with correct column names
import csv
with open(OUTPUT_CSV, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['ID', 'Label'])  # correct column names!
    for img_id, pred in zip(ids, preds):
        w.writerow([img_id, int(pred)])

print(f'\nDone!')

# Verify
import pandas as pd
df = pd.read_csv(OUTPUT_CSV)
print(df.head())
print('Shape:', df.shape)

  TTA pass 1/8 done
  TTA pass 2/8 done
  TTA pass 3/8 done
  TTA pass 4/8 done

Done!
                           ID  Label
0  1776338367036990_1_713.png      0
1  1776338367037012_2_151.png      0
2  1776338367037022_3_367.png      0
3  1776338367037030_4_184.png      0
4  1776338367037037_5_382.png      0
Shape: (5010, 2)


## Cell 12: Preview Predictions

In [44]:
import pandas as pd
df = pd.read_csv(OUTPUT_CSV)
print('Shape:', df.shape)
print('\nClass distribution:')
print(df['Label'].value_counts()) 
df.head(10)

Shape: (5010, 2)

Class distribution:
Label
1    3670
0    1340
Name: count, dtype: int64


,ID,Label
0,1776338367036990_1_713.png,0
1,1776338367037012_2_151.png,0
2,1776338367037022_3_367.png,0
3,1776338367037030_4_184.png,0
4,1776338367037037_5_382.png,0
5,1776338367037045_6_603.png,0
6,1776338367037052_7_635.png,0
7,1776338367037060_8_798.png,0
8,1776338367037068_9_288.png,0
9,1776338367037075_10_900.png,0
